# NOVA → YouTube: одноразовое подключение

Этот ноутбук нужен **один раз**. Он получает разрешение `youtube.upload` и возвращает refresh token. Ничего не сохраняется в Colab/Drive автоматически.

Перед запуском создай в Google Cloud OAuth client типа **TVs and Limited Input devices** и подготовь `Client ID` + `Client Secret`.

In [ ]:
import getpass, json, time, urllib.parse, urllib.request, urllib.error

DEVICE_ENDPOINT='https://oauth2.googleapis.com/device/code'
TOKEN_ENDPOINT='https://oauth2.googleapis.com/token'
SCOPE='https://www.googleapis.com/auth/youtube.upload'

def post_form(url, payload):
    data=urllib.parse.urlencode(payload).encode('utf-8')
    req=urllib.request.Request(url,data=data,headers={'Content-Type':'application/x-www-form-urlencoded'},method='POST')
    try:
        with urllib.request.urlopen(req,timeout=30) as r:
            return r.getcode(), json.loads(r.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        body=e.read().decode('utf-8',errors='replace')
        try: parsed=json.loads(body)
        except: parsed={'raw':body}
        return e.code, parsed

client_id=input('YOUTUBE_CLIENT_ID: ').strip()
client_secret=getpass.getpass('YOUTUBE_CLIENT_SECRET: ').strip()
status, device=post_form(DEVICE_ENDPOINT, {'client_id':client_id,'scope':SCOPE})
assert status==200, (status, device)

print('\nОткрой:', device['verification_url'])
print('Код:', device['user_code'])
print('Войди именно в аккаунт нужного YouTube-канала и нажми Allow.\n')

deadline=time.time()+int(device.get('expires_in',1800))
interval=int(device.get('interval',5))
while time.time()<deadline:
    time.sleep(interval)
    status, token=post_form(TOKEN_ENDPOINT, {
        'client_id':client_id,
        'client_secret':client_secret,
        'device_code':device['device_code'],
        'grant_type':'urn:ietf:params:oauth:grant-type:device_code'
    })
    if status==200 and token.get('refresh_token'):
        print('\nГОТОВО. Сохрани в GitHub Actions Secrets:')
        print('YOUTUBE_CLIENT_ID =', client_id)
        print('YOUTUBE_CLIENT_SECRET = [то значение, которое ввёл]')
        print('YOUTUBE_REFRESH_TOKEN =', token['refresh_token'])
        break
    err=token.get('error')
    if err=='authorization_pending':
        continue
    if err=='slow_down':
        interval += 5
        continue
    raise RuntimeError((status,token))
else:
    raise TimeoutError('Код истёк. Запусти ячейку ещё раз.')


## После получения токена

В GitHub → `magomedt149/nova-robot` → Settings → Secrets and variables → Actions добавь три секрета:

- `YOUTUBE_CLIENT_ID`
- `YOUTUBE_CLIENT_SECRET`
- `YOUTUBE_REFRESH_TOKEN`

После этого workflow **Daily YouTube Short - FREE** сможет выполнять официальный YouTube upload.